In [2]:
import random
import sqlite3
from datetime import datetime, timedelta

# ------------------------------------------------------------
# UK public transport database generator
# Creates: uk_transport_network.db
# Synthetic data only; it does not represent actual timetables.
# ------------------------------------------------------------

DB_NAME = "uk_transport_network.db"
SEED = 2026
NUM_JOURNEYS = 1250
START_DATETIME = datetime(2026, 1, 1)

random.seed(SEED)

STATIONS = [
    ("London St Pancras", "London"),
    ("Manchester Piccadilly", "North West"),
    ("Birmingham New Street", "Midlands"),
    ("Glasgow Central", "Scotland"),
    ("Leeds", "Yorkshire and the Humber"),
    ("Liverpool Lime Street", "North West"),
    ("Edinburgh Waverley", "Scotland"),
    ("Bristol Temple Meads", "South West"),
    ("Sheffield", "Yorkshire and the Humber"),
    ("Newcastle", "North East"),
    ("Nottingham", "East Midlands"),
    ("Cardiff Central", "Wales"),
    ("Southampton Central", "South East"),
    ("Brighton", "South East"),
    ("Norwich", "East of England"),
    ("York", "Yorkshire and the Humber"),
    ("Cambridge", "East of England"),
    ("Oxford", "South East"),
    ("Leicester", "East Midlands"),
    ("Derby", "East Midlands"),
    ("Coventry", "West Midlands"),
    ("Hull", "Yorkshire and the Humber"),
    ("Plymouth", "South West"),
    ("Swansea", "Wales"),
]

OPERATORS = [
    ("OP01", "Northern Rail"),
    ("OP02", "Great Western Railway"),
    ("OP03", "CrossCountry"),
    ("OP04", "ScotRail"),
    ("OP05", "Transport for Wales"),
    ("OP06", "East Midlands Railway"),
]

SERVICE_TYPES = ["Intercity", "Regional", "Local", "Commuter"]  # Nominal
OCCUPANCY_LEVELS = ["Low", "Medium", "High", "Full"]            # Ordinal
TICKET_TYPES = ["Advance", "Off-peak", "Anytime", "Season"]
AMENITIES = ["Wi-Fi", "Accessible toilet", "Bike space", "Power sockets"]
SOURCES = ["operator_feed", "station_feed", "manual_import"]


def create_database():
    conn = sqlite3.connect(DB_NAME)
    conn.execute("PRAGMA foreign_keys = ON")
    cur = conn.cursor()

    # Drop tables in dependency order to make reruns clean.
    cur.executescript("""
        DROP TABLE IF EXISTS journey_amenities;
        DROP TABLE IF EXISTS tickets;
        DROP TABLE IF EXISTS raw_journey_import;
        DROP TABLE IF EXISTS journeys;
        DROP TABLE IF EXISTS stations;
        DROP TABLE IF EXISTS operators;
    """)

    # Lookup tables keep repeated station and operator details in one place.
    cur.execute("""
        CREATE TABLE operators (
            operator_code TEXT PRIMARY KEY,
            operator_name TEXT NOT NULL UNIQUE
        )
    """)

    cur.execute("""
        CREATE TABLE stations (
            station_id INTEGER PRIMARY KEY,
            station_name TEXT NOT NULL UNIQUE,
            region TEXT NOT NULL CHECK (
                region IN (
                    'London', 'North West', 'Midlands',
                    'Scotland', 'Yorkshire and the Humber',
                    'North East', 'Wales', 'South East',
                    'East of England', 'East Midlands',
                    'West Midlands', 'South West'
                )
            )
        )
    """)

    # Main table: 1,250 journeys. origin and destination both reference stations.
    cur.execute("""
        CREATE TABLE journeys (
            journey_id INTEGER PRIMARY KEY,
            operator_code TEXT NOT NULL,
            origin_station_id INTEGER NOT NULL,
            destination_station_id INTEGER NOT NULL,
            service_type TEXT NOT NULL CHECK (
                service_type IN ('Intercity', 'Regional', 'Local', 'Commuter')
            ),
            occupancy_level TEXT NOT NULL CHECK (
                occupancy_level IN ('Low', 'Medium', 'High', 'Full')
            ),
            scheduled_departure TEXT NOT NULL,
            delay_minutes INTEGER CHECK (
                delay_minutes IS NULL OR delay_minutes >= 0
            ),
            fare_gbp REAL NOT NULL CHECK (fare_gbp >= 0),
            FOREIGN KEY (operator_code) REFERENCES operators(operator_code),
            FOREIGN KEY (origin_station_id) REFERENCES stations(station_id),
            FOREIGN KEY (destination_station_id) REFERENCES stations(station_id),
            CHECK (origin_station_id <> destination_station_id)
        )
    """)

    # A ticket is identified by its journey and seat; the same seat cannot be
    # entered twice for one journey. This is a composite primary key.
    cur.execute("""
        CREATE TABLE tickets (
            journey_id INTEGER NOT NULL,
            seat_reference TEXT NOT NULL,
            ticket_type TEXT NOT NULL CHECK (
                ticket_type IN ('Advance', 'Off-peak', 'Anytime', 'Season')
            ),
            price_gbp REAL NOT NULL CHECK (price_gbp >= 0),
            PRIMARY KEY (journey_id, seat_reference),
            FOREIGN KEY (journey_id) REFERENCES journeys(journey_id)
        )
    """)

    # Each journey can list an amenity once. This is a second composite key.
    cur.execute("""
        CREATE TABLE journey_amenities (
            journey_id INTEGER NOT NULL,
            amenity TEXT NOT NULL CHECK (
                amenity IN (
                    'Wi-Fi', 'Accessible toilet', 'Bike space', 'Power sockets'
                )
            ),
            PRIMARY KEY (journey_id, amenity),
            FOREIGN KEY (journey_id) REFERENCES journeys(journey_id)
        )
    """)

    # Staging/import table deliberately retains repeated source records and
    # missing values so the cleaning process can be demonstrated.
    cur.execute("""
        CREATE TABLE raw_journey_import (
            import_id INTEGER PRIMARY KEY,
            journey_id INTEGER NOT NULL,
            source_label TEXT NOT NULL,
            imported_delay_minutes INTEGER,
            FOREIGN KEY (journey_id) REFERENCES journeys(journey_id)
        )
    """)

    cur.executemany(
        "INSERT INTO operators (operator_code, operator_name) VALUES (?, ?)",
        OPERATORS,
    )

    cur.executemany(
        "INSERT INTO stations (station_id, station_name, region) VALUES (?, ?, ?)",
        [
            (station_id, station_name, region)
            for station_id, (station_name, region)
            in enumerate(STATIONS, start=1)
        ],
    )

    station_ids = list(range(1, len(STATIONS) + 1))

    for journey_id in range(1, NUM_JOURNEYS + 1):
        origin_id = random.choice(station_ids)
        destination_id = random.choice(station_ids)
        while destination_id == origin_id:
            destination_id = random.choice(station_ids)

        departure = START_DATETIME + timedelta(
            days=random.randint(0, 364),
            hours=random.randint(0, 23),
            minutes=random.choice([0, 15, 30, 45]),
        )

        # Deliberate missing values in 5% of journey delay records.
        delay = None if journey_id % 20 == 0 else random.randint(0, 60)
        fare = round(random.uniform(8.50, 180.00), 2)

        cur.execute("""
            INSERT INTO journeys (
                journey_id, operator_code, origin_station_id,
                destination_station_id, service_type, occupancy_level,
                scheduled_departure, delay_minutes, fare_gbp
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            journey_id,
            random.choice(OPERATORS)[0],
            origin_id,
            destination_id,
            random.choice(SERVICE_TYPES),
            random.choice(OCCUPANCY_LEVELS),
            departure.isoformat(sep=" "),
            delay,
            fare,
        ))

        # Insert one or two tickets; seat references are unique per journey.
        seats = random.sample(
            ["A01", "A02", "B01", "B02", "C01", "C02"],
            random.randint(1, 2),
        )
        for seat in seats:
            ticket_type = random.choice(TICKET_TYPES)
            cur.execute("""
                INSERT INTO tickets (
                    journey_id, seat_reference, ticket_type, price_gbp
                ) VALUES (?, ?, ?, ?)
            """, (
                journey_id,
                seat,
                ticket_type,
                round(fare * random.uniform(0.8, 1.2), 2),
            ))

        for amenity in random.sample(AMENITIES, random.randint(1, 3)):
            cur.execute("""
                INSERT INTO journey_amenities (journey_id, amenity)
                VALUES (?, ?)
            """, (journey_id, amenity))

        # Deliberate missing values in the raw import feed as well.
        imported_delay = None if journey_id % 25 == 0 else delay

        cur.execute("""
            INSERT INTO raw_journey_import (
                journey_id, source_label, imported_delay_minutes
            ) VALUES (?, ?, ?)
        """, (journey_id, random.choice(SOURCES), imported_delay))

    # Deliberately duplicate a sample of raw import rows. These are confined
    # to the staging table; the normalized journeys table remains unique.
    cur.execute("""
        INSERT INTO raw_journey_import (
            journey_id, source_label, imported_delay_minutes
        )
        SELECT journey_id, source_label, imported_delay_minutes
        FROM raw_journey_import
        WHERE journey_id % 40 = 0
    """)

    conn.commit()

    # Basic checks so the script confirms key requirements on completion.
    journey_count = cur.execute(
        "SELECT COUNT(*) FROM journeys"
    ).fetchone()[0]
    duplicate_raw_count = cur.execute("""
        SELECT COUNT(*) FROM (
            SELECT journey_id
            FROM raw_journey_import
            GROUP BY journey_id
            HAVING COUNT(*) > 1
        )
    """).fetchone()[0]

    print(f"Created {DB_NAME}")
    print(f"Journeys: {journey_count}")
    print(f"Journey IDs duplicated in raw staging data: {duplicate_raw_count}")

    conn.close()


if __name__ == "__main__":
    create_database()


Created uk_transport_network.db
Journeys: 1250
Journey IDs duplicated in raw staging data: 31
